In [ ]:
import os, json, re, textwrap, getpass, sys, warnings
from dataclasses import dataclass, field
from typing import Optional
from openai import OpenAI
from ddgs import DDGS
from rich.console import Console
from rich.table import Table
from rich.panel import Panel
from rich import box

warnings.filterwarnings("ignore", category=DeprecationWarning)

def _get_api_key() -> str:
    key = os.environ.get("OPENAI_API_KEY", "").strip()
    if key:
        return key
    try:
        from google.colab import userdata
        key = userdata.get("OPENAI_API_KEY") or ""
        if key.strip():
            return key.strip()
    except Exception:
        pass
    console = Console()
    console.print(
        "\n[bold cyan]OpenAI API Key required[/bold cyan]\n"
        "[dim]Your key will not be echoed and is never stored to disk.\n"
        "To skip this prompt in future runs, set the environment variable:\n"
        "  export OPENAI_API_KEY=sk-...[/dim]\n"
    )
    key = getpass.getpass("  Enter your OpenAI API key: ").strip()
    if not key:
        Console().print("[bold red]No API key provided — exiting.[/bold red]")
        sys.exit(1)
    return key

OPENAI_API_KEY = _get_api_key()
MODEL           = "gpt-4o-mini"
CONFIDENCE_LOW  = 0.55
CONFIDENCE_MED  = 0.80

client  = OpenAI(api_key=OPENAI_API_KEY)
console = Console()

@dataclass
class LLMResponse:
    question:    str
    answer:      str
    confidence:  float
    reasoning:   str
    sources:     list[str] = field(default_factory=list)
    researched:  bool = False
    raw_json:    dict = field(default_factory=dict)

In [ ]:
SYSTEM_UNCERTAINTY = """
You are an expert AI assistant that is HONEST about what it knows and doesn't know.
For every question you MUST respond with valid JSON only (no markdown, no prose outside JSON):

{
  "answer": "<your best answer — thorough, factual>",
  "confidence": <float 0.0-1.0>,
  "reasoning": "<explain WHY you are or aren't confident; mention specific knowledge gaps>"
}

Confidence scale:
  0.90-1.00 → very high: well-established fact, you are certain
  0.75-0.89 → high: strong knowledge, minor uncertainty
  0.55-0.74 → medium: plausible but you may be wrong, could be outdated
  0.30-0.54 → low: significant uncertainty, answer is a best guess
  0.00-0.29 → very low: mostly guessing, minimal reliable knowledge

Be CALIBRATED — do not always give high confidence. Genuinely reflect uncertainty
about recent events (after your knowledge cutoff), niche topics, numerical claims,
and anything that changes over time.
""".strip()

SYSTEM_SYNTHESIS = """
You are a research synthesizer. Given a question, a preliminary answer,
and web-search snippets, produce an improved final answer grounded in the evidence.
Respond in JSON only:

{
  "answer": "<improved, evidence-grounded answer>",
  "confidence": <float 0.0-1.0>,
  "reasoning": "<explain how the search evidence changed or confirmed the answer>"
}
""".strip()

def query_llm_with_confidence(question: str) -> LLMResponse:
    completion = client.chat.completions.create(
        model=MODEL,
        temperature=0.2,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": SYSTEM_UNCERTAINTY},
            {"role": "user",   "content": question},
        ],
    )
    raw = json.loads(completion.choices[0].message.content)

    return LLMResponse(
        question=question,
        answer=raw.get("answer", ""),
        confidence=float(raw.get("confidence", 0.5)),
        reasoning=raw.get("reasoning", ""),
        raw_json=raw,
    )

In [ ]:
def self_evaluate(response: LLMResponse) -> LLMResponse:
    critique_prompt = f"""
Review this answer and its stated confidence. Check for:
1. Logical consistency
2. Whether the confidence matches the actual quality of the answer
3. Any factual errors you can spot

Question: {response.question}

Proposed answer: {response.answer}
Stated confidence: {response.confidence}
Stated reasoning: {response.reasoning}

Respond in JSON:
{{
  "revised_confidence": <float — adjust if the self-check changes your view>,
  "critique": "<brief critique of the answer quality>",
  "revised_answer": "<improved answer, or repeat original if fine>"
}}
""".strip()

    completion = client.chat.completions.create(
        model=MODEL,
        temperature=0.1,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": "You are a rigorous self-critic. Respond in JSON only."},
            {"role": "user",   "content": critique_prompt},
        ],
    )
    ev = json.loads(completion.choices[0].message.content)

    response.confidence = float(ev.get("revised_confidence", response.confidence))
    response.answer     = ev.get("revised_answer", response.answer)
    response.reasoning += f"\n\n[Self-Eval Critique]: {ev.get('critique', '')}"
    return response


def web_search(query: str, max_results: int = 5) -> list[dict]:
    results = DDGS().text(query, max_results=max_results)
    return list(results) if results else []


def research_and_synthesize(response: LLMResponse) -> LLMResponse:
    console.print(f"  [yellow]🔍 Confidence {response.confidence:.0%} is low — triggering auto-research...[/yellow]")

    snippets = web_search(response.question)
    if not snippets:
        console.print("  [red]No search results found.[/red]")
        return response

    formatted = "\n\n".join(
        f"[{i+1}] {s.get('title','')}\n{s.get('body','')}\nURL: {s.get('href','')}"
        for i, s in enumerate(snippets)
    )

    synthesis_prompt = f"""
Question: {response.question}

Preliminary answer (low confidence): {response.answer}

Web search snippets:
{formatted}

Synthesize an improved answer using the evidence above.
""".strip()

    completion = client.chat.completions.create(
        model=MODEL,
        temperature=0.2,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": SYSTEM_SYNTHESIS},
            {"role": "user",   "content": synthesis_prompt},
        ],
    )
    syn = json.loads(completion.choices[0].message.content)

    response.answer      = syn.get("answer", response.answer)
    response.confidence  = float(syn.get("confidence", response.confidence))
    response.reasoning  += f"\n\n[Post-Research]: {syn.get('reasoning', '')}"
    response.sources     = [s.get("href", "") for s in snippets if s.get("href")]
    response.researched  = True
    return response

In [ ]:
def confidence_label(score: float) -> tuple[str, str]:
    if score >= 0.90: return "🟢", "Very High"
    if score >= 0.75: return "🔵", "High"
    if score >= 0.55: return "🟡", "Medium"
    if score >= 0.30: return "🟠", "Low"
    return "🔴", "Very Low"

def _step(msg: str) -> None:
    console.print(f"  [dim]⏳ {msg}[/dim]", end="\r")

def uncertainty_aware_query(question: str) -> LLMResponse:
    _step("Stage 1 · Generating answer + confidence...")
    resp = query_llm_with_confidence(question)
    console.print("  [dim]✓ Stage 1 complete[/dim]        ")

    _step("Stage 2 · Self-evaluating response...")
    resp = self_evaluate(resp)
    console.print("  [dim]✓ Stage 2 complete[/dim]        ")

    if resp.confidence < CONFIDENCE_LOW:
        _step("Stage 3 · Searching the web for evidence...")
        resp = research_and_synthesize(resp)
        console.print("  [dim]✓ Stage 3 complete[/dim]        ")

    return resp


def display_response(resp: LLMResponse) -> None:
    emoji, label = confidence_label(resp.confidence)
    conf_color = (
        "green" if resp.confidence >= 0.75
        else "yellow" if resp.confidence >= 0.55
        else "red"
    )

    console.print(Panel(
        f"[bold white]{resp.question}[/bold white]",
        title="[bold cyan]❓ Question[/bold cyan]",
        border_style="cyan",
    ))

    console.print(Panel(
        textwrap.fill(resp.answer, width=90),
        title="[bold green]💬 Answer[/bold green]",
        border_style="green",
    ))

    bar_len  = 30
    filled   = int(resp.confidence * bar_len)
    bar      = "█" * filled + "░" * (bar_len - filled)
    console.print(
        f"\n  {emoji} Confidence: [{conf_color}]{bar}[/{conf_color}]"
        f" [{conf_color}]{resp.confidence:.0%} — {label}[/{conf_color}]"
    )

    console.print(Panel(
        resp.reasoning.strip(),
        title="[bold yellow]🧠 Reasoning & Self-Evaluation[/bold yellow]",
        border_style="yellow",
    ))

    if resp.researched and resp.sources:
        src_text = "\n".join(f"  [{i+1}] {url}" for i, url in enumerate(resp.sources[:5]))
        console.print(Panel(
            src_text,
            title="[bold magenta]🌐 Auto-Research Sources[/bold magenta]",
            border_style="magenta",
        ))

    console.print()

In [8]:
DEMO_QUESTIONS = [
    "What is the speed of light in a vacuum?",
    "What were the main causes of the 2008 global financial crisis?",
    "What is the latest version of Python released in 2025?",
    "What is the current population of Tokyo as of 2025?",
]

def run_comparison_table(questions: list[str]) -> None:
    console.rule("[bold cyan]UNCERTAINTY-AWARE LLM — BATCH RUN[/bold cyan]")
    results = []

    for i, q in enumerate(questions, 1):
        console.print(f"\n[bold]Question {i}/{len(questions)}:[/bold] {q}")
        r = uncertainty_aware_query(q)
        display_response(r)
        results.append(r)

    console.rule("[bold cyan]SUMMARY TABLE[/bold cyan]")
    tbl = Table(box=box.ROUNDED, show_lines=True, highlight=True)
    tbl.add_column("#",          style="dim", width=3)
    tbl.add_column("Question",   max_width=40)
    tbl.add_column("Confidence", justify="center", width=12)
    tbl.add_column("Level",      justify="center", width=10)
    tbl.add_column("Researched", justify="center", width=10)

    for i, r in enumerate(results, 1):
        emoji, label = confidence_label(r.confidence)
        col = "green" if r.confidence >= 0.75 else "yellow" if r.confidence >= 0.55 else "red"
        tbl.add_row(
            str(i),
            textwrap.shorten(r.question, 55),
            f"[{col}]{r.confidence:.0%}[/{col}]",
            f"{emoji} {label}",
            "✅ Yes" if r.researched else "—",
        )

    console.print(tbl)


def interactive_mode() -> None:
    console.rule("[bold cyan]INTERACTIVE MODE[/bold cyan]")
    console.print("  Type any question. Type [bold]quit[/bold] to exit.\n")
    while True:
        q = console.input("[bold cyan]You ▶[/bold cyan] ").strip()
        if q.lower() in ("quit", "exit", "q"):
            console.print("Goodbye!")
            break
        if not q:
            continue
        resp = uncertainty_aware_query(q)
        display_response(resp)


if __name__ == "__main__":
    console.print(Panel(
        "[bold white]Uncertainty-Aware LLM Tutorial[/bold white]\n"
        "[dim]Confidence Estimation · Self-Evaluation · Auto-Research[/dim]",
        border_style="cyan",
        expand=False,
    ))

    run_comparison_table(DEMO_QUESTIONS)

    console.print("\n")
    interactive_mode()

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

OpenAI API Key required
Your key will not be echoed and is never stored to disk.
To skip this prompt in future runs, set the environment variable:
  export OPENAI_API_KEY=sk-...

╭─────────────────────────────────────────────────────────╮
│ Uncertainty-Aware LLM Tutorial                          │
│ Confidence Estimation · Self-Evaluation · Auto-Research │
╰─────────────────────────────────────────────────────────╯

──────────────────────────────────────── UNCERTAINTY-AWARE LLM — BATCH RUN ────────────────────────────────────────

Question 1/4: What is the speed of light in a vacuum?

...

──────────────────────────────────────────────── INTERACTIVE MODE ─────────────────────────────────────────────────

Type any question. Type quit to exit.

You ▶

exit


Goodbye!